In [1]:
## init mongo db and fiftyone connection
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [ ]:
import fiftyone.brain as fob
from sklearn.preprocessing import normalize
import plotly.express as px
import skdim
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import ot
from sklearn.manifold import TSNE
import cv2
from fiftyone import ViewField as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import random

## renders plotly properly in a html instance. 
#import plotly.io as pio
#pio.renderers.default = "notebook"


In [3]:
## Load dataset and views from mongodb 
dataset = fo.load_dataset("dugong")

## load the views
nc_view = dataset.load_saved_view("New_Caledonia")
wp_view = dataset.load_saved_view("West_Papua")

In [4]:
## get embeddings for each view and normalize it 
nc_emb = nc_view.values("full_embeddings")
wp_emb = wp_view.values("full_embeddings")

## convert to array
nc_emb = np.array(nc_emb)
wp_emb = np.array(wp_emb)

## normalize firsrt
nc_norm = normalize(nc_emb)
wp_norm = normalize(wp_emb)

## CREATE VIEWS OF THE DATASET THAT ARE REPRESENTATIVE OF GEOGRAPHICAL LOCATIONS
## NOTE : Mantasandy is being excluded from further analysis because
## it only has 3 images inside the folder. 
view_GAM = wp_view.match(
    F("subregion")=="GAM"
)
view_FRIWEN = wp_view.match(
    F("subregion")=="FRIWEN"
)
view_MANTASANDY = wp_view.match(
    F("subregion")=="MANTASANDY"
)
view_UM = wp_view.match(
    F("subregion")=="UM"
)
gam_emb = view_GAM.values('full_embeddings')
friwen_emb = view_FRIWEN.values('full_embeddings')
um_emb = view_UM.values('full_embeddings')

norm_gam_emb = normalize(gam_emb)
norm_friwen_emb = normalize(friwen_emb)
norm_um_emb = normalize(um_emb)

print("Number of images per geographical location")
print(f"WP- GAM: {len(view_GAM)}")
print(f"WP - FRIWEN: {len(view_FRIWEN)}")
print(f"WP - MANTASANDY: {len(view_MANTASANDY)}")
print(f"WP -UM: {len(view_UM)}")
print(f"NC: {len(nc_view)}")
print(f"WP (total): {len(wp_view)}")

datasets_l ={'WP': wp_norm, 
             'NC' : nc_norm,
            'WP_GAM':norm_gam_emb,
            'WP_friwen':norm_friwen_emb,
            'WP_mant':norm_um_emb}



Number of images per geographical location
WP- GAM: 512
WP - FRIWEN: 779
WP - MANTASANDY: 3
WP -UM: 745
NC: 716
WP (total): 2039


In [7]:
session = fo.launch_app(dataset,auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


In [9]:
next(iter(dataset.take(1)))

<SampleView: {
    'id': '69add89acfd942e1c6b5ddf2',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/WP/UM/UM_M6/images/MAN_P4_UM_M6_F2_GSP_DJI_0224-658070ac304b8_178.jpeg',
    'tags': ['UM_M6', 'UM', 'WP'],
    'metadata': <ImageMetadata: {
        'size_bytes': 906162,
        'mime_type': 'image/jpeg',
        'width': 4096,
        'height': 2160,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 18, 889000),
    'last_modified_at': datetime.datetime(2026, 3, 20, 13, 30, 32, 31000),
    'region': 'WP',
    'subregion': 'UM',
    'mission_name': 'UM_M6',
    'sea_state': 1,
    'turbidity_global': 1,
    'turbidity_local': 'Unknown',
    'sun_glitter': '25-50',
    'cloud_reflection': '0-0',
    'habitat_type': 'sand',
    'background_complexity': 'medium',
    'coral': 'P',
    'sand': 'P',
    'dense_seagrass': 'P',
    'open_sea': 'A',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        

In [54]:
from pathlib import Path
import random
filepath = dataset.values("filepath")

## iter and extract
extract_first_word = []
for fil in filepath:
    extract_first_word.append(Path(fil).stem.split("_")[0])

len(extract_first_word)

2755

In [70]:
data = random.choices(extract_first_word, k=10)

# def contains_gh(word):
#     return bool(re.search(r"GH", word))

# result = [contains_gh(word) for word in data]
# print(result)



def map_to_gh(word):
    return "GH" if re.search(r"^GH", word) else word

result = list(map(map_to_gh, extract_first_word))
print("Unique labels found it.")
print(pd.Series(result).unique())


Unique labels found it.
<ArrowStringArray>
['GH', 'FPLAN', 'MAN']
Length: 3, dtype: str


In [ ]:
import re 
## add field to dataset

filepath = dataset.values("filepath")

## iter and extract
## very simple extraction of the first sentence 
extract_first_word = []
for fil in filepath:
    extract_first_word.append(Path(fil).stem.split("_")[0])

## we convert everything that is GH as from New Calodonia, and therefore we rename if to GH
def map_to_gh(word):
    return "GH" if re.search(r"^GH", word) else word

result = list(map(map_to_gh, extract_first_word))
print("Unique labels found it.")
print(pd.Series(result).unique())


# create a mapping dictionary: {filepath: tag}
# We use absolute filepaths to be 100% safe
filepaths = dataset.values("filepath")
tag_map = dict(zip(filepaths, result))

# 3. Add the field to the dataset (if it doesn't exist yet)
if not dataset.has_sample_field("flight_plan"):
    dataset.add_sample_field("flight_plan", fo.StringField)

# 4. Bulk update the values
dataset.set_values("flight_plan", tag_map, key_field="filepath")



Unique labels found it.
<ArrowStringArray>
['GH', 'FPLAN', 'MAN']
Length: 3, dtype: str


In [74]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['Flight_226', 'NC', 'NC'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 3, 24, 19, 11, 8, 395000),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections': [
     